# **Module 3.1: Trajectory Optimization** 🏁📈

---

**What this notebook does:** So far our policies have driven the car directly, choosing a steering
angle and a speed 100 times a second. They learned to *survive* the track (Learn to Drive),
to drive it *cleanly* (Learn to Race), and to *react* to obstacles. None of them were asked
the question a race engineer asks: **what is the fastest way around this track?**

In this notebook the policy stops being the driver and becomes the **race engineer**. At every
decision it chooses *where on the track* the car should be (a sideways offset from the
centerline) and *how fast* it should go, and a classic pure-pursuit controller does the
actual steering. The path the car traces lap after lap **is** the racing line the policy has
learned, and we will watch it evolve from "follow the centerline" to a fast, smooth line
as training goes on.

📌 Same from-scratch PPO algorithm as every other notebook in this series (network, GAE,
clipped loss), unchanged. Only the action meaning and the reward change.


<img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/mod3.png" width="750">

---

---

## 📝 Student TODOs — Complete These Before Running Training

This notebook has **6 TODOs** to complete. TODOs 5 and 6 are the PPO pieces from
Learn to Race; paste in your own working versions if you have them. Everything else is given to you.

| # | TODO | Location | What You Build |
|---|---|---|---|
| **TODO 1** | Offset to target point on the track | Section 3, `get_target_point` | Pure-pursuit target |
| **TODO 2** | Curvature preview in the observation | Section 4, `_process_obs` | Look-ahead of the track shape |
| **TODO 3** | Scale actions to offset and speed | Section 4, `_scale_action` | The new action space |
| **TODO 4** | Lap-time reward | Section 4, wrapper `step()` | Reward design |
| **TODO 5** | Actor-critic forward pass | Section 5, `ActorCritic.forward` | Same as Learn to Race |
| **TODO 6** | Compute GAE advantages | Section 5.2, `compute_gae` | Same as Learn to Race |

> **📌 Tip:** Read each section's explanation carefully before attempting its TODO. The
> math and logic you need is explained right above the corresponding code cell.

---

## 📋 How This Works

| | Learn to Drive / Learn to Race | Learn the Racing Line |
|---|---|---|
| **Policy outputs** | Steering angle and speed | Sideways offset from the centerline and a target speed |
| **Who steers** | The policy, directly | Pure pursuit, chasing the point the policy chose |
| **Decision rate** | 100 Hz (every sim step) | 20 Hz (each decision is held for 5 sim steps) |
| **Reward** | Progress plus shaping terms | Metres of progress per decision, wall-margin and grip penalties, and a crash penalty |
| **Physics** | Whatever the simulator allows | Exceeding the tyres' real grip limit counts as a spin-out (a crash), like on the real car |
| **How we judge it** | Episode reward | **Lap time**, and the **path drawn on the map** |


<img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/traj_blockdiagram.png" width="700">

**In code:** one call to `RacingLineWrapper.step(action)` is one policy decision.
1. `_scale_action` turns the network output into an offset `d` (metres) and a speed `v` (m/s).
2. The line moves smoothly toward `d`: 50% of the way each decision, and never more than
   0.15 m per decision.
3. For the next 5 sim steps, `get_target_point` places a point a short distance ahead on the
   track, shifted sideways by the offset. `pure_pursuit_steer` computes the steering angle
   toward it, and `[steer, speed]` is sent to the simulator. The car's sideways acceleration
   is tracked every sim step; if it goes past the tyres' grip, the car has spun out.
4. The wrapper adds up the progress made in those 5 steps, computes the reward, and returns
   the new observation to the policy.

Every eval lap is recorded as an `(x, y, speed)` trace, so we can **see** the learned
racing line on the track map and how it improves over training.


<img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/traj.gif" width="950">
---

---

# **Section 1 — Concepts** 💡

## 1.1 What Is a Racing Line? 🏎️

The fastest way around a track is almost never the centerline. Cornering speed is limited by
how much sideways grip the tyres have:

$$v_{\max} \;=\; \sqrt{\frac{a_{\text{lat,max}}}{\kappa}}$$

- $\kappa = 1/R$ is the curvature of the path the car drives (a tighter turn has a larger $\kappa$)
- $a_{\text{lat,max}} \approx \mu g$ is the most sideways acceleration the tyres can give

So the **larger the radius** you drive through a corner, the faster you can take it. That is
why racing drivers go **outside, then inside at the apex, then outside again**: using the
whole track width straightens the corner. The line, the braking point and the speed profile
are all linked.

There is a second, competing effect: a **shorter** path also saves time. When a car is limited
by its top speed rather than by grip, the shortest path (the inside of every corner) wins.
The fastest real line balances the two.

## 1.2 Classic Optimization vs Learning It 🧮

The classic approach (minimum-curvature or minimum-time optimization) solves for the line
offline, using a vehicle model and an assumed friction limit. The F1TENTH map files come with
exactly such a line, which we use as our **reference raceline**.

> **Reference raceline:** the racing line that comes with the F1TENTH Spielberg map files
> (`Spielberg_raceline.csv`). It was computed offline by a minimum-time optimizer, using a vehicle
> model and a conservative grip limit (its top speed is 8 m/s). We drive it with the same pure
> pursuit controller, so it is a fair "classic optimization" benchmark for our learned line.

The RL approach needs no model: it discovers the line *and* the speed profile from driving.

## 1.3 Our MDP 🎯

**State** (121 numbers, all scaled to $[0, 1]$): the 108 lidar beams, speed, steering angle,
the current sideways offset $e_y$ from the centerline, and the **curvature of the track ahead**
(10 samples over the next 20 m).

**Action:** two numbers $a_0, a_1 \in [-1, 1]$, turned into an offset and a speed:

$$d \;=\; a_0 \cdot d_{\max} \qquad\qquad v \;=\; v_{\min} + \frac{a_1 + 1}{2}\,\big(v_{\max} - v_{\min}\big)$$

- $d$ is **where on the track width** the car should be: metres left (+) or right (−) of the centerline, up to $d_{\max} = 0.85$ m
- $v$ is the **target speed**, between $v_{\min} = 2$ and $v_{\max} = 10$ m/s

**Reward** for one decision, written the same way as in the earlier notebooks:

$$r_t = \Delta_{\text{progress}} - 0.5\max(0,\ 0.4 - d_{\min}) - 0.2\max(0,\ a_{\text{lat}} - 8.2) - 10\cdot\mathbb{1}[\text{collision or spin-out}]$$

- **Progress** $\Delta_{\text{progress}}$: metres gained **along the centerline** this decision (the same progress signal as every earlier notebook, now in metres)
- **Wall penalty** (same as Learn to Race): $d_{\min}$ is the closest lidar distance; it costs a little whenever the car is closer than 0.4 m to a wall
- **Grip penalty** (new): $a_{\text{lat}}$ is the sideways acceleration (speed times yaw rate); it costs reward above 8.2 m/s² (80% of grip), so the policy keeps grip in reserve
- **Crash penalty** (same as before, now also for a spin-out, Section 1.4): the episode ends, which also loses all the progress the rest of it would have earned
- **Dropped from Learn to Race:** the speed bonus (progress already rewards speed), the steer penalty (the offset filter keeps the line smooth) and the centerline lateral-error penalty (a racing line has to leave the centerline)

Episodes have a **fixed length** (30 s), so collecting the most progress in that time is the
same as **minimizing lap time**. A line that cuts a corner still earns full centerline
progress, just sooner.


## 1.4 Keeping It Physically Real 🛞

The simulator's simple tyre model never runs out of grip: it would let the car corner at
several times what real tyres can hold, and an RL agent will happily exploit that with sharp,
flat-out turns that would spin a real car. We handle this **in the reward and the episode
rules**, so the policy has to learn to respect grip by itself:

$$a_{\text{lat}} \;=\; v\,\omega \qquad\qquad \text{spin-out if } a_{\text{lat}} > 1.1\,\mu g \approx 11.3 \text{ m/s}^2$$

- $a_{\text{lat}}$ is the car's sideways acceleration: speed $v$ times yaw rate $\omega$, low-pass filtered over about 0.2 s so that brief sim chatter does not count
- $\mu g \approx 10.3$ m/s² is the tyres' real grip (gym friction $\mu = 1.05$). The reference raceline itself is optimized to run right at this limit, so we allow a 10% tolerance for short transients
- Going past $1.1\,\mu g$ is a **spin-out**: it is treated exactly like a crash (penalty, episode ends)
- Going past 80% of grip (about 8.2 m/s²) costs a little reward every decision, so the policy keeps a safety margin

With these rules, the only way to carry speed through a corner is a **larger radius**:
enter wide, clip the apex, exit wide. That is the shape of a real racing line, and it is the
same physics the reference raceline was optimized for.

## 1.5 Why 20 Hz Decisions? ⏱️

PPO plans roughly $1/(1-\gamma)$ decisions ahead. With $\gamma = 0.99$:

| Decision rate | How far ahead the policy plans |
|---|---|
| 100 Hz (every sim step) | About 100 steps, which is **1 second**: too short to set up for a corner |
| **20 Hz** (each decision held for 5 sim steps) | About 100 decisions, which is **5 seconds**: enough to brake *before* the hairpin |





<img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/traj_opt.png" width="550">


---

---

# **Section 2 — Setup** 🔧

In [ ]:
# ── Install all dependencies ──────────────────────────────────────────────────
!pip install -q git+https://github.com/f1tenth/f1tenth_gym.git@dev-humble
!pip install -q stable-baselines3[extra]
!pip install -q wandb imageio imageio-ffmpeg

print("Installation complete!")

## 2.2 Verify Imports 📦

In [ ]:
# ── Verify everything imports correctly ───────────────────────────────────────
import os
import copy
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
from collections import deque
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.collections import LineCollection
from IPython.display import clear_output, Image, display
import wandb
import f1tenth_gym

# Vectorized environment utilities only; the PPO algorithm itself is ours.
from stable_baselines3.common.vec_env import (
    SubprocVecEnv, DummyVecEnv, VecMonitor, VecNormalize
)

print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("All good, let's find the racing line!")

## 2.3 Meet the Track 🗺️

We load Spielberg's occupancy map, its **centerline** and the **reference raceline** (with its
speed profile). The plotting helpers defined here are reused for every visualization in this
notebook, including **zoomed views of the three sharpest corners**, where racing lines differ
the most.

> **Reference raceline:** the racing line that comes with the F1TENTH Spielberg map files
> (`Spielberg_raceline.csv`). It was computed offline by a minimum-time optimizer, using a vehicle
> model and a conservative grip limit (its top speed is 8 m/s). We drive it with the same pure
> pursuit controller, so it is a fair "classic optimization" benchmark for our learned line.

In [ ]:
# ── Load the track + plotting helpers ─────────────────────────────────────────
from f1tenth_gym.envs.track import Track

MAP_NAME = "Spielberg"
TRACK    = Track.from_track_name(MAP_NAME)      # also caches the map files
CL, RL   = TRACK.centerline, TRACK.raceline
CL_XY    = np.column_stack([CL.xs, CL.ys])
RL_XY    = np.column_stack([RL.xs, RL.ys])
TRACK_LENGTH = float(CL.length)
HALF_WIDTH   = 1.1                              # metres, from Spielberg_centerline.csv

_occ  = np.array(TRACK.occupancy_map)
_spec = TRACK.spec
_res  = _spec.resolution
_ox, _oy = _spec.origin[0], _spec.origin[1]
_EXTENT  = [_ox, _ox + _occ.shape[1] * _res, _oy, _oy + _occ.shape[0] * _res]

# The 3 sharpest corners (largest |curvature|, at least 25 m apart), used for zoom views
_ss, _ks = np.asarray(CL.ss), np.asarray(CL.ks)
CORNER_S = []
for i in np.argsort(-np.abs(_ks)):
    if all(min(abs(_ss[i] - s), TRACK_LENGTH - abs(_ss[i] - s)) > 25 for s in CORNER_S):
        CORNER_S.append(float(_ss[i]))
    if len(CORNER_S) == 3:
        break
CORNER_XY = [TRACK.frenet_to_cartesian(s, 0.0, 0.0)[:2] for s in CORNER_S]

V_PLOT_MIN, V_PLOT_MAX = 2.0, 10.0              # colour scale for speed (m/s)
CL_COLOR, RL_COLOR = "#777777", "#00bcd4"       # centerline = gray, reference raceline = cyan


def track_ax(ax, zoom_xy=None, zoom=7.0):
    """Draw the occupancy map on ax; optionally zoom to a box around zoom_xy."""
    ax.imshow(_occ, cmap="gray", origin="lower", extent=_EXTENT, vmin=0, vmax=255)
    if zoom_xy is None:
        ax.set_xlim(CL_XY[:, 0].min() - 3, CL_XY[:, 0].max() + 3)
        ax.set_ylim(CL_XY[:, 1].min() - 3, CL_XY[:, 1].max() + 3)
    else:
        ax.set_xlim(zoom_xy[0] - zoom, zoom_xy[0] + zoom)
        ax.set_ylim(zoom_xy[1] - zoom, zoom_xy[1] + zoom)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])


def speed_line(ax, path, lw=2.5):
    """Draw an (N, 3) [x, y, speed] path on ax, coloured by speed."""
    pts  = path[:, :2].reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, cmap="turbo", norm=plt.Normalize(V_PLOT_MIN, V_PLOT_MAX), linewidth=lw)
    lc.set_array(path[:-1, 2])
    ax.add_collection(lc)
    return lc


def plain_line(ax, xy, color, label=None, lw=1.4, ls="-"):
    ax.plot(xy[:, 0], xy[:, 1], color=color, lw=lw, ls=ls, label=label)


plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "legend.fontsize": 10})

# ── Show the track, the centerline and the reference raceline ─────────────────
RL_PATH = np.column_stack([RL_XY, np.asarray(RL.vxs)])
fig, axs = plt.subplots(1, 4, figsize=(20, 5.5), gridspec_kw={"width_ratios": [2, 1, 1, 1]})
track_ax(axs[0]); plain_line(axs[0], CL_XY, CL_COLOR, "centerline", ls="--")
lc = speed_line(axs[0], RL_PATH)
for k, (cx, cy) in enumerate(CORNER_XY):
    axs[0].annotate(f"C{k+1}", (cx, cy), color="red", fontsize=13, weight="bold")
axs[0].set_title(f"{MAP_NAME}: reference raceline coloured by its speed (track {TRACK_LENGTH:.0f} m)")
axs[0].legend(loc="lower left")
for k, ax in enumerate(axs[1:]):
    track_ax(ax, CORNER_XY[k]); plain_line(ax, CL_XY, CL_COLOR, ls="--"); speed_line(ax, RL_PATH)
    ax.set_title(f"Corner C{k+1} (s = {CORNER_S[k]:.0f} m)")
fig.colorbar(lc, ax=axs, fraction=0.015, label="speed (m/s)")
plt.show()
print(f"Reference raceline speed range: {RL.vxs.min():.1f} to {RL.vxs.max():.1f} m/s")

---

# **Section 3 — The Path Follower: Pure Pursuit** 🧭

## 3.1 From "Where to Be" to a Steering Angle

The policy never steers. It picks an **offset** $d$ (metres left or right of the centerline)
and a **speed** $v$. We turn that into a point the car should chase:

1. Read the car's current progress $s$ along the centerline (from the gym's Frenet pose).
2. Look ahead by a distance that grows with speed:

$$L_d \;=\; L_{\min} + k_v\, v$$

3. The target point is the place on the track at progress $s + L_d$, shifted sideways by $d$.
   The track object converts that (progress, offset) pair back to world coordinates with
   `track.frenet_to_cartesian(s + L_d, d, 0)`, giving the target $(x_t, y_t)$.

4. Pure pursuit steers along the circular arc through the target (the same law as the opponent
   car in Dynamic Obstacle Avoidance):

$$y_{\text{local}} \;=\; -\Delta x \,\sin\theta \;+\; \Delta y\, \cos\theta$$

$$\kappa \;=\; \frac{2\, y_{\text{local}}}{L_d^{\,2}} \qquad\qquad \delta \;=\; \arctan\!\big(\ell_{\text{wb}}\, \kappa\big)$$

- $\Delta x, \Delta y$: the target's position relative to the car, and $\theta$ is the car's heading
- $y_{\text{local}}$: how far the target sits to the car's left (+) or right (−)
- $\ell_{\text{wb}} = 0.33$ m is the wheelbase, and $\delta$ is the steering angle sent to the sim

Because the target slides along the track as the car moves, holding the same $d$ traces a
smooth line parallel to the centerline; changing $d$ moves the car across the track.

## 3.2 Implementation ⚙️

In [ ]:
# ── Pure pursuit path follower ────────────────────────────────────────────────
WHEELBASE      = 0.33   # metres, matches the gym's default vehicle params (lf + lr)
LOOKAHEAD_MIN  = 0.8    # metres of lookahead at standstill
LOOKAHEAD_GAIN = 0.05   # seconds: extra lookahead per m/s of speed


def get_target_point(track, s, offset, speed):
    """
    Point the car should chase: LOOKAHEAD ahead of progress s along the
    centerline, shifted sideways by `offset` metres. Returns (tx, ty, lookahead).
    """
    lookahead = LOOKAHEAD_MIN + LOOKAHEAD_GAIN * speed

    # ═════════════════════════════════════════════════════════════════
    # ✏️  TODO 1 — Offset to Target Point on the Track
    # ═════════════════════════════════════════════════════════════════

    # Move ahead of s by the lookahead (wrapping at the lap length), then
    # convert that track position, shifted sideways by the offset, back to
    # world (x, y) with the track's Frenet-to-Cartesian conversion.
    # Return: tx, ty, lookahead

    # ─────────────────────────────────────────────────────────────────





    # ═════════════════════════════════════════════════════════════════
    pass  # remove once you've implemented the steps above

def pure_pursuit_steer(x, y, theta, tx, ty, lookahead, max_steer):
    """Steering angle (rad) that drives the car along the arc through (tx, ty)."""
    dx, dy    = tx - x, ty - y
    local_y   = -dx * np.sin(theta) + dy * np.cos(theta)
    curvature = 2.0 * local_y / (lookahead ** 2)
    return float(np.clip(np.arctan(WHEELBASE * curvature), -max_steer, max_steer))


print("Pure pursuit follower defined.")

---

# **Section 4 — The Racing-Line Environment** 🧩

## 4.1 Observation, Action and Reward

**Observation** (121 values, all scaled to between 0 and 1). The first 110 values are
**exactly the observation from Learn to Drive and Learn to Race**; the last 11 are new:

| Index | What it is | How it is scaled |
|---|---|---|
| 0 to 107 | Lidar scan (108 beams) | Divided by 10 m |
| 108 | Speed | Divided by the top speed (10 m/s) |
| 109 | Steering angle | Full left = 0, straight = 0.5, full right = 1 |
| 110 | Offset from the centerline | Right track edge = 0, centerline = 0.5, left edge = 1 |
| 111 to 120 | **Curvature preview**: track curvature 2, 4, 6 ... 20 m ahead | Curvature of −1.5 to +1.5 per metre mapped to 0 to 1 |

The offset and the curvature preview are the only additions, and they are what make this a
trajectory problem: the lidar only sees about 10 m of walls, but braking for a
hairpin at 10 m/s has to start earlier than that. Giving the policy the **shape of the track
ahead** is like a driver who has learned the circuit.

**Action:** the same two numbers in $[-1, 1]$ as in the earlier notebooks. The second one is
still the speed (now 2 to 10 m/s). The first one is now an **offset** of up to 0.85 m either
side of the centerline instead of a steering angle (the track is 2.2 m wide; the wall-margin
penalty keeps the car off the walls). Each decision is held for 5 sim steps while pure pursuit
steers, and the line moves toward the chosen offset smoothly (50% of the way per decision, at
most 0.15 m), fast enough to swing from the outside to the apex of a hairpin, but without
snapping.

In a corner tighter than the offset itself (the hairpin's centerline radius is only about
0.74 m), an inside offset larger than the radius has no meaning, so inside offsets are capped
at 70% of the local radius. Offsets to the outside of a corner are not limited.

**Reward:**

| Term | Value | Why |
|---|---|---|
| **Progress** | +1.0 per metre gained along the centerline | Fixed-length episodes, so more metres means a faster lap |
| **Wall penalty** | −0.5 per metre that the car is inside 0.4 m of a wall | Same term as Learn to Race: a small push away from the walls |
| **Grip penalty** (new) | −0.2 per m/s² of sideways acceleration above 8.2 m/s² (80% of grip) | Prefers lines that do not ride right at the grip limit |
| **Crash penalty** | −10, and the episode ends | Hitting a wall, or a spin-out: sideways acceleration past 11.3 m/s² (grip plus 10%) |

There is **no speed bonus and no centerline penalty**, so the agent is free to use the whole
track, and speed only pays off if it turns into progress.

Training episodes start at a **random point on the track** (the `RL_RANDOM_STATIC` reset) and
last 30 s (600 decisions), so every corner gets practised. Evaluation laps always start from
the start/finish line.

## 4.2 Implementation ⚙️

In [ ]:
# ── Racing-line configuration ─────────────────────────────────────────────────
MAX_OFFSET     = 0.85    # metres, max offset from the centerline (track half-width is 1.1 m)
OFFSET_FILTER  = 0.5     # each decision the line moves 50% of the way to the chosen offset (low-pass)
MAX_OFFSET_RATE = 0.15   # ...and at most this many metres per decision (3 m/s of sideways drift)
INSIDE_LIMIT   = 0.7     # inside a corner, the offset may use at most 70% of the local radius
V_MIN, V_MAX   = 2.0, 10.0   # m/s (above 10 m/s pure pursuit starts to chatter in this car model)
ACTION_REPEAT  = 5       # sim steps per decision (100 Hz sim, 20 Hz policy)
PREVIEW_N      = 10      # curvature samples ahead
PREVIEW_DS     = 2.0     # metres between samples (20 m preview)
KAPPA_SCALE    = 1.5     # 1/m, curvature normalization
EPISODE_STEPS  = 600     # decisions per training episode (30 s)

# Tyre grip (Section 1.4)
MU_G           = 1.0489 * 9.81   # real tyre grip limit: gym friction mu x g, about 10.3 m/s^2
SPIN_OUT_LIMIT = 1.1 * MU_G      # about 11.3 m/s^2: past this the car has spun out (10% tolerance)
A_LAT_FILTER   = 0.05    # low-pass on sideways acceleration (time constant about 0.2 s)

# Reward coefficients (CRASH_PENALTY and GRIP_COEF are tuned by the sweep in Section 6.3)
PROGRESS_COEF       = 1.0    # progress reward, per metre gained along the centerline
WALL_PROX_COEF      = 0.5    # wall penalty (same term as Learn to Race), per metre inside the threshold
WALL_PROX_THRESHOLD = 0.4    # metres
GRIP_COEF           = 0.2    # grip penalty (new), per m/s^2 of sideways acceleration above the threshold
GRIP_THRESHOLD      = 0.8 * MU_G   # about 8.2 m/s^2: the grip penalty starts at 80% of grip
CRASH_PENALTY       = 10.0   # crash penalty: hitting a wall or spinning out ends the episode


class RacingLineWrapper(gym.Env):
    """
    Observation (121-dim, all in [0,1]): lidar(108), speed, steer, offset e_y,
                                          curvature preview(10)
    Action (2-dim, in [-1,1]):  [0] offset from centerline, [1] target speed
    Each step() = ACTION_REPEAT sim steps of pure pursuit toward the chosen line.
    """

    def __init__(self, map_name=MAP_NAME, num_beams=108, render_mode=None):
        super().__init__()
        from f1tenth_gym.envs.env_config import (
            EnvConfig, SimulationConfig, ObservationConfig, ResetConfig,
        )
        from f1tenth_gym.envs.lidar.config import LiDARConfig
        from f1tenth_gym.envs.observation import ObservationType
        from f1tenth_gym.envs.reset import ResetStrategy
        from f1tenth_gym.envs.integrators import IntegratorType
        from f1tenth_gym.envs.dynamic_models import DynamicModel

        env_config = EnvConfig(
            map_name=map_name,
            num_agents=1,
            simulation_config=SimulationConfig(
                timestep=0.01,
                integrator=IntegratorType.RK4,
                dynamics_model=DynamicModel.ST,
                max_laps=100,                     # we end episodes ourselves
            ),
            observation_config=ObservationConfig(type=ObservationType.DIRECT),
            lidar_config=LiDARConfig(num_beams=num_beams, range_max=10.0, noise_std=0.0),
            reset_config=ResetConfig(strategy=ResetStrategy.RL_RANDOM_STATIC),
            render_enabled=(render_mode is not None),
        )
        self.env = gym.make("f1tenth_gym:f1tenth-v0", config=env_config, render_mode=render_mode)

        self.track       = self.env.unwrapped.track
        self.track_len   = float(self.track.centerline.length)
        self._ss         = np.asarray(self.track.centerline.ss, dtype=np.float64)
        self._ks         = np.asarray(self.track.centerline.ks, dtype=np.float64)
        self.num_beams   = num_beams
        self.lidar_clip  = 10.0
        self.max_steer   = float(self.env.action_space.high[0][0])
        self.max_steps   = EPISODE_STEPS
        self.dt          = 0.01

        self.observation_space = gym.spaces.Box(
            low=0.0, high=1.0, shape=(num_beams + 3 + PREVIEW_N,), dtype=np.float32
        )
        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)

    # ----------------------------------------------------------

    def _curvature_at(self, s):
        """Centerline curvature (1/m) at progress s (array OK, wraps around the lap)."""
        return np.interp(np.mod(s, self.track_len), self._ss, self._ks, period=self.track_len)

    def _process_obs(self, raw_obs):
        agent = raw_obs["agent_0"]
        scan  = np.array(agent["scan"], dtype=np.float32) / self.lidar_clip
        state = agent["std_state"]
        s, ey = float(agent["frenet_pose"][0]), float(agent["frenet_pose"][1])

        speed_n = np.clip(float(state[3]) / V_MAX, 0.0, 1.0)
        steer_n = np.clip((float(state[2]) + self.max_steer) / (2 * self.max_steer), 0.0, 1.0)
        ey_n    = np.clip((ey / HALF_WIDTH + 1.0) / 2.0, 0.0, 1.0)

        # ═════════════════════════════════════════════════════════════
        # ✏️  TODO 2 — Curvature Preview in the Observation
        # ═════════════════════════════════════════════════════════════

        # Sample the centerline curvature at PREVIEW_N points spaced PREVIEW_DS
        # metres ahead of s (use self._curvature_at), then map it from
        # [-KAPPA_SCALE, KAPPA_SCALE] to [0, 1] and clip -> kappa_n

        # ─────────────────────────────────────────────────────────────





        # ═════════════════════════════════════════════════════════════
        return np.concatenate([
            scan, np.array([speed_n, steer_n, ey_n], dtype=np.float32), kappa_n
        ]).astype(np.float32)

    def _scale_action(self, action):

        # ═════════════════════════════════════════════════════════════
        # ✏️  TODO 3 — Scale Actions to Offset and Speed
        # ═════════════════════════════════════════════════════════════

        # action[0], action[1] are in [-1, 1]. Map them to physical units:
        #   offset: scale to [-MAX_OFFSET, +MAX_OFFSET] metres
        #   speed:  map to [V_MIN, V_MAX] m/s (same mapping as Learn to Drive/Race)
        # Return: offset, speed

        # ─────────────────────────────────────────────────────────────





        # ═════════════════════════════════════════════════════════════
        pass  # remove once you've implemented the steps above

    # ----------------------------------------------------------

    def reset(self, seed=None, options=None):
        raw_obs, info     = self.env.reset(seed=seed, options=options)
        self.raw_obs      = raw_obs
        self.current_step = 0
        self.prev_s       = float(raw_obs["agent_0"]["frenet_pose"][0])
        # the line starts wherever the car is (clipped to the allowed offset range)
        self.offset       = float(np.clip(raw_obs["agent_0"]["frenet_pose"][1], -MAX_OFFSET, MAX_OFFSET))
        self.a_lat        = 0.0         # filtered sideways acceleration (m/s^2)
        self.path         = []          # (x, y, speed, lateral accel) every sim step, for the plots
        return self._process_obs(raw_obs), info

    def step(self, action):
        target_offset, speed = self._scale_action(action)
        # the line moves smoothly toward the chosen offset: low-pass filtered,
        # and never more than MAX_OFFSET_RATE per decision
        self.offset += float(np.clip(OFFSET_FILTER * (target_offset - self.offset),
                                     -MAX_OFFSET_RATE, MAX_OFFSET_RATE))
        offset = self.offset
        delta_progress, collision, spun_out, done, truncated = 0.0, False, False, False, False
        a_lat_peak = 0.0

        # Hold the decision for ACTION_REPEAT sim steps; pure pursuit steers each one
        for _ in range(ACTION_REPEAT):
            agent = self.raw_obs["agent_0"]
            state = agent["std_state"]
            x, y, theta = float(state[0]), float(state[1]), float(state[4])
            s = float(agent["frenet_pose"][0])

            # In a corner tighter than the offset itself (the hairpin's centerline radius is
            # only about 0.74 m), an inside offset bigger than the radius has no meaning, so
            # inside offsets are capped at INSIDE_LIMIT of the local radius. Outside is free.
            k = float(self._curvature_at(s + LOOKAHEAD_MIN + LOOKAHEAD_GAIN * speed))
            target_off = INSIDE_LIMIT / k if offset * k > INSIDE_LIMIT else offset

            tx, ty, lookahead = get_target_point(self.track, s, target_off, speed)
            steer = pure_pursuit_steer(x, y, theta, tx, ty, lookahead, self.max_steer)
            self.raw_obs, _, done, truncated, info = self.env.step(
                np.array([[steer, speed]], dtype=np.float32)
            )

            agent = self.raw_obs["agent_0"]
            s_new = float(agent["frenet_pose"][0])
            ds = s_new - self.prev_s                              # progress this sim step,
            if ds < -self.track_len / 2: ds += self.track_len     # with lap wrap-around
            if ds >  self.track_len / 2: ds -= self.track_len
            delta_progress   += ds
            self.prev_s = s_new
            st = agent["std_state"]
            a_lat_now = abs(float(st[3]) * float(st[5]))          # sideways accel = speed x yaw rate
            self.a_lat += A_LAT_FILTER * (a_lat_now - self.a_lat) # low-pass filtered
            a_lat_peak = max(a_lat_peak, self.a_lat)
            self.path.append((float(st[0]), float(st[1]), float(st[3]), self.a_lat))
            collision = bool(agent.get("collision", False))
            spun_out  = self.a_lat > SPIN_OUT_LIMIT               # beyond the tyres' grip: a real car slides out
            if collision or spun_out or done:
                break
        self.current_step += 1
        min_scan = float(np.min(self.raw_obs["agent_0"]["scan"]))

        # ═════════════════════════════════════════════════════════════
        # ✏️  TODO 4 — Lap-Time Reward
        # ═════════════════════════════════════════════════════════════

        # Build the reward from the four terms in Section 4.1:
        #   progress reward : delta_progress scaled by PROGRESS_COEF
        #   wall penalty    : grows as min_scan drops below WALL_PROX_THRESHOLD
        #   grip penalty    : grows as a_lat_peak rises above GRIP_THRESHOLD
        #   crash penalty   : on a collision or a spin-out, the reward becomes
        #                     -CRASH_PENALTY and the episode ends (done = True)

        # ─────────────────────────────────────────────────────────────
        reward = 0.0





        # ═════════════════════════════════════════════════════════════
        if self.current_step >= self.max_steps:
            truncated = True

        info = {
            "speed":        float(self.raw_obs["agent_0"]["std_state"][3]),
            "collision":    collision,
            "spun_out":     spun_out,
            "ds":           delta_progress,
            "offset":       offset,
            "target_speed": speed,
            "min_scan":     min_scan,
            "a_lat":        a_lat_peak,
        }
        return self._process_obs(self.raw_obs), float(reward), bool(done), bool(truncated), info

    def render(self):
        return self.env.render()

    def close(self):
        self.env.close()


print("RacingLineWrapper defined!")
print("Remember: complete all TODOs before running training!")

## 4.3 Baselines: How Fast Is "Fast"? ⏱️

Before training, we time three reference laps from the start/finish line:

| Baseline | What it is |
|---|---|
| **Centerline at 7 m/s** | What our *untrained* policy does: its average action is offset 0 and the middle speed, 7 m/s (expect a spin-out in the first tight corner) |
| **Best constant speed** | The fastest constant speed that completes a full centerline lap without a crash or a spin-out |
| **Reference raceline** | The line that ships with the map files, driven with its own speed profile (it starts on its own line) |

> **Reference raceline:** the racing line that comes with the F1TENTH Spielberg map files
> (`Spielberg_raceline.csv`). It was computed offline by a minimum-time optimizer, using a vehicle
> model and a conservative grip limit (its top speed is 8 m/s). We drive it with the same pure
> pursuit controller, so it is a fair "classic optimization" benchmark for our learned line.

`run_lap` is our evaluation tool for the rest of the notebook: it drives one lap with any
policy and returns the lap time and the full `(x, y, speed)` trace. A lap that crashes is
marked **DNF** (did not finish).

In [ ]:
# ── Lap timing + baselines ────────────────────────────────────────────────────
START_POSE = np.array([[CL.xs[0], CL.ys[0], CL.yaws[0]]], dtype=np.float64)
V_MID = (V_MIN + V_MAX) / 2                    # speed of the untrained policy's mean action


def run_lap(env, policy_fn, max_decisions=3000):
    """Drive ONE lap from the start line. policy_fn(obs) -> action in [-1, 1]^2.
    Returns dict(lap_time [s, or None for a DNF], path (N,3), crashed, dist [m])."""
    obs, _ = env.reset(options={"poses": START_POSE})
    env.max_steps = 10**9                       # no episode cap while timing a lap
    dist, n, crashed = 0.0, 0, False
    while dist < env.track_len and n < max_decisions:
        obs, _, _, _, info = env.step(policy_fn(obs))
        dist += info["ds"]; n += 1
        if info["collision"] or info["spun_out"]:
            crashed = True
            break
    env.max_steps = EPISODE_STEPS
    done_lap = (not crashed) and dist >= env.track_len
    return {"lap_time": len(env.path) * env.dt if done_lap else None,
            "path": np.array(env.path), "crashed": crashed, "dist": dist}


def lap_str(r):
    return f"{r['lap_time']:.2f} s" if r["lap_time"] else f"DNF after {r['dist']:.0f} m"


def constant_policy(offset, speed):
    a = np.array([offset / MAX_OFFSET, (speed - V_MIN) / (V_MAX - V_MIN) * 2 - 1], dtype=np.float32)
    return lambda obs: a


def raceline_lap(env, lmin=0.8, lgain=0.05):
    """Pure pursuit on the reference raceline, using its speed profile."""
    rx, ry, rv = RL_XY[:, 0], RL_XY[:, 1], np.asarray(RL.vxs)
    seg = np.hypot(np.diff(rx, append=rx[0]), np.diff(ry, append=ry[0]))
    env.reset(options={"poses": np.array([[rx[0], ry[0], RL.yaws[0]]])})
    raw, idx, path, dist, a_lat = env.raw_obs, 0, [], 0.0, 0.0
    prev_s = float(raw["agent_0"]["frenet_pose"][0])
    while dist < env.track_len and len(path) < 30000:
        st = raw["agent_0"]["std_state"]; x, y, th = float(st[0]), float(st[1]), float(st[4])
        win = (idx + np.arange(40)) % len(rx)
        idx = int(win[np.argmin(np.hypot(rx[win] - x, ry[win] - y))])
        v, ld = float(rv[idx]), lmin + lgain * float(rv[idx])
        j, acc = idx, 0.0
        while acc < ld:
            acc += seg[j]; j = (j + 1) % len(rx)
        steer = pure_pursuit_steer(x, y, th, rx[j], ry[j], ld, env.max_steer)
        raw, _, _, _, _ = env.env.step(np.array([[steer, v]], dtype=np.float32))
        s_new = float(raw["agent_0"]["frenet_pose"][0]); ds = s_new - prev_s
        if ds < -env.track_len / 2: ds += env.track_len
        if ds >  env.track_len / 2: ds -= env.track_len
        dist += ds; prev_s = s_new
        st = raw["agent_0"]["std_state"]
        a_lat += A_LAT_FILTER * (abs(float(st[3]) * float(st[5])) - a_lat)
        path.append((float(st[0]), float(st[1]), float(st[3]), a_lat))
        if raw["agent_0"]["collision"] or a_lat > SPIN_OUT_LIMIT:
            return {"lap_time": None, "path": np.array(path), "crashed": True, "dist": dist}
    return {"lap_time": len(path) * env.dt, "path": np.array(path), "crashed": False, "dist": dist}


base_env = RacingLineWrapper()
BASELINES = {}
BASELINES[f"Centerline at {V_MID:.0f} m/s"] = run_lap(base_env, constant_policy(0.0, V_MID))

best = None
for v in np.arange(3.0, V_MAX + 0.01, 0.5):
    r = run_lap(base_env, constant_policy(0.0, float(v)))
    print(f"  centerline at {v:4.1f} m/s: {lap_str(r)}")
    if r["lap_time"] is None:
        break                                    # faster constant speeds only get worse
    best = (float(v), r)
if best is not None:
    BASELINES[f"Best constant ({best[0]:.1f} m/s)"] = best[1]
BASELINES["Reference raceline"] = raceline_lap(base_env)
CENTER_KEY = f"Centerline at {V_MID:.0f} m/s"

print()
for name, r in BASELINES.items():
    print(f"  {name:<26} lap time: {lap_str(r)}")

---

# **Section 5 — PPO Algorithm (Unchanged)** 🧠

The network, GAE, and clipped loss are identical to our from-scratch PPO in Learn to
Race; only the environment changed.

> **📌 Note:** Since this is unchanged from your earlier notebooks, feel free to paste in your own working PPO implementation (network, GAE, clipped loss) from Learn to Race here instead of rewriting it, if you already have one you trust.

In [ ]:
# ── Actor-Critic Network ──────────────────────────────────────────────────────
# Two INDEPENDENT [64,64] MLPs, no shared trunk. Sharing weights between the
# actor and critic would let policy gradients and value gradients interfere
# with the same parameters, hurting both objectives. Orthogonal
# initialization with a small gain on the action head keeps initial actions
# near zero, which is important for early training stability.

def _orthogonal_init(layer, gain):
    nn.init.orthogonal_(layer.weight, gain=gain)
    nn.init.constant_(layer.bias, 0.0)

class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        # Actor: fully independent MLP, own weights
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.actor_mean = nn.Linear(hidden, act_dim)

        # Critic: fully independent MLP, own weights (no sharing with the actor)
        self.critic_body = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.critic = nn.Linear(hidden, 1)

        # Learnable log standard deviation (state-independent)
        self.log_std = nn.Parameter(torch.zeros(act_dim))

        # Orthogonal init gains: sqrt(2) for hidden layers, 0.01 for the
        # action output (near-zero initial actions), 1.0 for the value output
        for layer in self.actor:
            if isinstance(layer, nn.Linear): _orthogonal_init(layer, gain=2 ** 0.5)
        for layer in self.critic_body:
            if isinstance(layer, nn.Linear): _orthogonal_init(layer, gain=2 ** 0.5)
        _orthogonal_init(self.actor_mean, gain=0.01)
        _orthogonal_init(self.critic, gain=1.0)

    def forward(self, obs):
        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 5 — Actor-Critic Forward Pass
        # ═══════════════════════════════════════════════════════════════

        # Pass obs through the actor to get the action mean and through the
        # critic to get the state value, then build a Normal distribution
        # from the mean and the learned log-std.
        # Return: dist, value

        # ───────────────────────────────────────────────────────────────





        # ═══════════════════════════════════════════════════════════════
        pass  # remove once you've implemented the steps above

    @torch.no_grad()
    def act(self, obs, deterministic=False):
        """Sample an action (or take the mean if deterministic)."""
        dist, value = self.forward(obs)
        action = dist.mean if deterministic else dist.sample()
        log_prob = dist.log_prob(action).sum(-1)
        return action, log_prob, value


def greedy_policy(net):
    """Deterministic policy function for run_lap()."""
    def _pi(obs):
        with torch.no_grad():
            a, _, _ = net.act(torch.tensor(obs, dtype=torch.float32, device=next(net.parameters()).device),
                              deterministic=True)
        return np.clip(a.cpu().numpy(), -1.0, 1.0)
    return _pi


print('ActorCritic network defined (separate actor/critic MLPs, orthogonal init).')

## 5.1 Environment Builders 🏭

In [ ]:
# ── Environment factory ───────────────────────────────────────────────────────

def make_env(rank, seed=0):
    """Returns a function that creates one RacingLineWrapper instance."""
    def _init():
        env = RacingLineWrapper()
        env.reset(seed=seed + rank)   # different seed per env = different random start points
        return env
    return _init


def make_training_env(num_envs):
    fns = [make_env(i) for i in range(num_envs)]
    env = DummyVecEnv(fns) if num_envs == 1 else SubprocVecEnv(fns)
    env = VecMonitor(env)
    env = VecNormalize(env, norm_obs=False, norm_reward=True, clip_reward=10.0, gamma=0.99)
    return env


print("Environment builders defined successfully.")

## 5.2 GAE: Generalized Advantage Estimation 📊

In [ ]:
# ── GAE — Generalized Advantage Estimation ────────────────────────────────────
def compute_gae(rewards, values, dones, last_value, gamma=0.99, lam=0.95):
    """
    rewards, values, dones : tensors of shape (T, N)  [T steps, N envs]
    last_value             : tensor (N,), V(s_T) for bootstrapping
    Returns advantages and returns, both (T, N).
    """
    T, N = rewards.shape
    advantages = torch.zeros_like(rewards)
    last_adv = torch.zeros(N, device=rewards.device)

    # ═══════════════════════════════════════════════════════════════
    # ✏️  TODO 6 — Compute GAE Advantages
    # ═══════════════════════════════════════════════════════════════

    # Loop backwards from t = T-1 to 0. At each step, take the next value
    # (last_value at the end), discount it only if the episode didn't end,
    # compute the TD residual, then add the decayed advantage from step t+1
    # and store the result in advantages[t].

    # ───────────────────────────────────────────────────────────────





    # ═══════════════════════════════════════════════════════════════

    returns = advantages + values
    return advantages, returns


print('compute_gae defined.')

---

# **Section 6 — Training** 🏋️

One "step" here is one **decision**, which is 5 sim steps or 0.05 s of driving.

Two additions keep this training stable, compared with the earlier notebooks:

| Addition | Why |
|---|---|
| **Learning-rate and entropy annealing** (both down to 0 over training) | Explore many lines early; late updates get smaller and the policy commits to one line instead of being knocked off course |
| **Best-lap checkpoint** | Every eval lap is timed; the network with the fastest completed lap is saved and used for evaluation |

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
NUM_ENVS    = 4             # parallel environments
TOTAL_STEPS = 1_000_000     # decisions (5M sim steps, about 40 to 50 min on a Colab T4 runtime)
DEVICE      = "cuda"
EVAL_FREQ   = 25_000        # a timed eval lap every N decisions; these laps build the plots

print("Training config:")
print(f"  Map:    {MAP_NAME}")
print(f"  Envs:   {NUM_ENVS}")
print(f"  Steps:  {TOTAL_STEPS:,} decisions")
print(f"  Device: {DEVICE}")

## 6.2 WandB Login 🔑

In [ ]:
# ── WandB login ───────────────────────────────────────────────────────────────
# Choose option 1 or 2, then paste your WANDB_API key
# Get it from: wandb.ai/authorize
# WANDB_API_KEY = "your_key_here"
# wandb.login(key=WANDB_API_KEY)

wandb.login()

## 6.3 Reward Sweep 🔬 *(Run Before Full Training)*

Two reward knobs shape the line:

| Knob | Too low | Too high |
|---|---|---|
| `CRASH_PENALTY` | Reckless driving, frequent crashes | Timid driving, slow laps |
| `GRIP_COEF` | The line rides right at the grip limit | The car corners too gently and gives away lap time |

The sweep trains a short 60k-decision policy for every combination and ranks them by **lap
time**, the real objective, not by reward (the swept coefficients are part of the reward, so
rewards from different runs are not comparable). A DNF lap scores `SWEEP_FAIL_TIME`.

**How to use:**
1. ✅ Complete **TODOs 1 to 6** first; the sweep uses your wrapper and your PPO pieces
2. Edit the `parameters` lists below to the values you want to compare
3. Run the cell; WandB launches one run per combination
4. Open the sweep link that prints; results update live
5. Put the best values in the configuration cell (Section 4.2), re-run it, then train in 6.5

In [ ]:
# ── WandB Reward Sweep ────────────────────────────────────────────────────────
sweep_config = {
    "method": "grid",
    "metric": {"name": "eval/lap_time", "goal": "minimize"},
    "parameters": {
        "crash_penalty":  {"values": [5.0, 10.0, 20.0]},
        "grip_coef":   {"values": [0.1, 0.3]},
    }
}

SWEEP_STEPS     = 60_000
SWEEP_ENVS      = 2
SWEEP_N_STEPS   = 1024      # rollout length per env per update
SWEEP_EPOCHS    = 10
SWEEP_MINIBATCH = 256
SWEEP_FAIL_TIME = 200.0     # score for a DNF lap


def sweep_train():
    global CRASH_PENALTY, GRIP_COEF

    run = wandb.init(reinit=True)     # a fresh WandB run for every combination
    cfg = wandb.config
    print(f"  crash_penalty={cfg.crash_penalty} | grip_coef={cfg.grip_coef}")

    # The wrapper reads these module-level constants inside step(), so write
    # this run's values to them before the envs are built.
    _orig = (CRASH_PENALTY, GRIP_COEF)
    CRASH_PENALTY = cfg.crash_penalty
    GRIP_COEF  = cfg.grip_coef

    s_train = s_eval = None
    try:
        torch.manual_seed(42); np.random.seed(42)
        device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")

        # DummyVecEnv (same process) so the constants set above are the ones the envs read
        s_train = DummyVecEnv([make_env(i) for i in range(SWEEP_ENVS)])
        s_train = VecMonitor(s_train)
        s_train = VecNormalize(s_train, norm_obs=False, norm_reward=True,
                               clip_reward=10.0, gamma=0.99)
        s_eval  = RacingLineWrapper()

        obs_dim = s_train.observation_space.shape[0]
        act_dim = s_train.action_space.shape[0]
        net = ActorCritic(obs_dim, act_dim).to(device)
        opt = optim.Adam(net.parameters(), lr=3e-4, eps=1e-5)

        obs = torch.tensor(s_train.reset(), dtype=torch.float32, device=device)
        for _update in range(SWEEP_STEPS // (SWEEP_N_STEPS * SWEEP_ENVS)):
            b_obs_l, b_act_l, b_logp_l, b_val_l, b_rew_l, b_done_l = [], [], [], [], [], []
            for _t in range(SWEEP_N_STEPS):
                action, logp, value = net.act(obs)
                next_obs, reward, dones, _ = s_train.step(np.clip(action.cpu().numpy(), -1.0, 1.0))
                b_obs_l.append(obs); b_act_l.append(action)
                b_logp_l.append(logp); b_val_l.append(value)
                b_rew_l.append(torch.tensor(reward, dtype=torch.float32, device=device))
                b_done_l.append(torch.tensor(dones, dtype=torch.float32, device=device))
                obs = torch.tensor(next_obs, dtype=torch.float32, device=device)

            with torch.no_grad():
                _, last_value = net(obs)
            values = torch.stack(b_val_l)
            advantages, returns = compute_gae(torch.stack(b_rew_l), values, torch.stack(b_done_l),
                                              last_value, gamma=0.99, lam=0.95)
            b_obs  = torch.stack(b_obs_l).reshape(-1, obs_dim)
            b_act  = torch.stack(b_act_l).reshape(-1, act_dim)
            b_logp = torch.stack(b_logp_l).reshape(-1)
            b_adv  = advantages.reshape(-1)
            b_ret  = returns.reshape(-1)

            idxs = np.arange(b_obs.shape[0])
            for _epoch in range(SWEEP_EPOCHS):
                np.random.shuffle(idxs)
                for start in range(0, len(idxs), SWEEP_MINIBATCH):
                    mb = idxs[start:start + SWEEP_MINIBATCH]
                    dist, value = net(b_obs[mb])
                    new_logp = dist.log_prob(b_act[mb]).sum(-1)
                    entropy  = dist.entropy().sum(-1).mean()
                    mb_adv   = b_adv[mb]
                    mb_adv   = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)
                    ratio    = torch.exp(new_logp - b_logp[mb])
                    policy_loss = -torch.min(ratio * mb_adv,
                                             torch.clamp(ratio, 0.8, 1.2) * mb_adv).mean()
                    value_loss  = ((value - b_ret[mb]) ** 2).mean()
                    loss = policy_loss + 0.5 * value_loss - 0.001 * entropy
                    opt.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(net.parameters(), 0.5)
                    opt.step()

        # ── Timed deterministic lap = the sweep metric ───────────────────────
        lap = run_lap(s_eval, greedy_policy(net))
        lap_time = lap["lap_time"] if lap["lap_time"] is not None else SWEEP_FAIL_TIME
        wandb.log({
            "eval/lap_time":      lap_time,
            "eval/lap_completed": float(lap["lap_time"] is not None),
            "eval/lap_distance":  lap["dist"],
            "eval/avg_speed":     float(np.mean(lap["path"][:, 2])) if len(lap["path"]) else 0.0,
        })
        print(f"    lap: {lap_str(lap)}")
    finally:
        if s_train is not None: s_train.close()
        if s_eval  is not None: s_eval.close()
        CRASH_PENALTY, GRIP_COEF = _orig   # restore your chosen values
        run.finish()

# ── Launch ───────────────────────────────────────────────────────────────────
n_combos = (len(sweep_config["parameters"]["crash_penalty"]["values"]) *
            len(sweep_config["parameters"]["grip_coef"]["values"]))
sweep_id = wandb.sweep(sweep_config, project="f1tenth-rl-racing-line-sweep")
print(f"Sweep: {n_combos} combinations of {SWEEP_STEPS:,} decisions each")
print(f"View at: https://wandb.ai/{wandb.Api().default_entity}/f1tenth-rl-racing-line-sweep/sweeps/{sweep_id}")
wandb.agent(sweep_id, function=sweep_train, count=None)
print()
print("✅ Sweep complete. Open the WandB link above to compare results.")
print("   Put the best values in the configuration cell (Section 4.2), re-run it, then train in 6.5.")

## 6.4 Training Dashboard 📈

| Panel | What to look for |
|---|---|
| **ep_reward** | Rises as the car covers more ground per episode |
| **crash rate (last 100 episodes)** | Falls toward zero; this is the *current* rate, not a total since the start |
| **eval lap time** | Falls; red crosses mark DNF eval laps |
| **avg speed** | Rises, then levels off once corners limit it |
| **policy std** | Falls as the policy becomes more certain |
| **total loss** | Stays bounded; see the note below |

> **Why does the loss jump around?** In RL the loss is not a progress score the way it is in
> supervised learning. Every update trains on *new* data from the current policy, and the
> value targets move as the policy improves, so the loss naturally rises and falls. Judge
> training by **lap time and crash rate**. Learning-rate annealing makes the late updates
> smaller, which also calms the loss toward the end.

In [ ]:
# ── Live Inline Training Plots ────────────────────────────────────────────────
_live = {"steps": [], "rew": [], "crash": [], "speed": [], "loss": [], "std": []}
LAP_HISTORY = []    # one entry per eval lap: dict(step, lap_time, dist, path)


def draw_live_plots(total_steps):
    if len(_live["steps"]) < 2:
        return
    BG = "#0d1117"
    fig = plt.figure(figsize=(15, 7), facecolor=BG)
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.3)
    panels = [
        (gs[0, 0], _live["steps"], _live["rew"],   "ep_reward ↑",                       "#00bcd4"),
        (gs[0, 1], _live["steps"], _live["crash"], "crash rate, last 100 episodes ↓",   "#f44336"),
        (gs[1, 0], _live["steps"], _live["speed"], "avg_speed m/s ↑",                   "#69ff47"),
        (gs[1, 1], _live["steps"], _live["std"],   "policy std ↓",                      "#9c27b0"),
        (gs[1, 2], _live["steps"], _live["loss"],  "total_loss",                        "#e91e63"),
    ]
    for spec, xs, data, title, color in panels:
        ax = fig.add_subplot(spec); ax.set_facecolor("#1a1a2e")
        ax.plot(xs, data, color=color, lw=2, marker="o", markersize=3)
        ax.fill_between(xs, data, min(data), alpha=0.15, color=color)
        ax.set_title(title, color="white", fontsize=11, pad=4)
        ax.set_xlabel("Decisions", color="#aaa", fontsize=9)
        ax.tick_params(colors="#aaa", labelsize=9)
        for spine in ax.spines.values(): spine.set_edgecolor("#333")

    # eval lap time, with DNF laps marked
    ax = fig.add_subplot(gs[0, 2]); ax.set_facecolor("#1a1a2e")
    ok  = [(h["step"], h["lap_time"]) for h in LAP_HISTORY if h["lap_time"] is not None]
    dnf = [h["step"] for h in LAP_HISTORY if h["lap_time"] is None]
    if ok:
        xs, ys = zip(*ok)
        ax.plot(xs, ys, color="#ff9800", lw=2, marker="o", markersize=4)
    if dnf:
        y_dnf = max(ys) if ok else 0
        ax.plot(dnf, [y_dnf] * len(dnf), "x", color="#ff1744", markersize=9, mew=2, label="DNF")
        ax.legend(fontsize=9, facecolor="#1a1a2e", labelcolor="white")
    ax.set_title("eval lap time (s) ↓", color="white", fontsize=11, pad=4)
    ax.set_xlabel("Decisions", color="#aaa", fontsize=9)
    ax.tick_params(colors="#aaa", labelsize=9)
    for spine in ax.spines.values(): spine.set_edgecolor("#333")

    step = _live["steps"][-1]
    wandb_url = wandb.run.url if wandb.run else "not connected"
    fig.suptitle(f"PPO from Scratch: Racing Line  |  Step {step:,} / {total_steps:,} "
                 f"({step / total_steps * 100:.1f}%)   |   WandB: {wandb_url}",
                 color="white", fontsize=11, x=0.02, ha="left")
    clear_output(wait=True)
    plt.show()
    plt.close(fig)

print("Live plot helper defined.")

## 6.5 Train 🚀

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PPO from Scratch — Racing Line Training (Run This Single Cell)
# ══════════════════════════════════════════════════════════════════════════════
# Identical from-scratch PPO algorithm to Learn to Race, plus learning-rate
# annealing and a best-lap checkpoint. Every EVAL_FREQ decisions we time one
# deterministic lap and keep its path in LAP_HISTORY; Section 8 turns those
# laps into the visualizations.
# ──────────────────────────────────────────────────────────────────────────────
N_STEPS       = 1024        # decisions collected per env per update
N_EPOCHS      = 10
MINIBATCH     = 256
GAMMA         = 0.99
GAE_LAMBDA    = 0.95
CLIP_EPS      = 0.2
ENT_COEF      = 0.003       # starting entropy bonus (annealed to 0): many lines are worth trying early on
VF_COEF       = 0.5
LR            = 3e-4        # starting learning rate, annealed linearly to 0
MAX_GRAD      = 0.5

torch.manual_seed(42)
np.random.seed(42)

run = wandb.init(
    project = "f1tenth-rl-lecture1",
    name    = f"ppo_scratch_racing_line_{MAP_NAME}_{TOTAL_STEPS//1000}k",
    config  = dict(n_steps=N_STEPS, n_epochs=N_EPOCHS, gamma=GAMMA, gae_lambda=GAE_LAMBDA,
                   clip_eps=CLIP_EPS, ent_coef=ENT_COEF, lr=LR, lr_schedule="linear",
                   map=MAP_NAME, num_envs=NUM_ENVS, max_offset=MAX_OFFSET, v_max=V_MAX,
                   action_repeat=ACTION_REPEAT, crash_penalty=CRASH_PENALTY,
                   wall_prox_coef=WALL_PROX_COEF, grip_coef=GRIP_COEF, grip_threshold=GRIP_THRESHOLD),
)
wandb.define_metric("global_step")
wandb.define_metric("*", step_metric="global_step")
print(f"🔗 WandB: {run.url}\n")

train_env = make_training_env(NUM_ENVS)
eval_env  = RacingLineWrapper()            # timed laps from the start line
obs_dim   = train_env.observation_space.shape[0]
act_dim   = train_env.action_space.shape[0]

device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
net    = ActorCritic(obs_dim, act_dim).to(device)
opt    = optim.Adam(net.parameters(), lr=LR, eps=1e-5)
print(net)
print(f"Parameters: {sum(p.numel() for p in net.parameters()):,}\n")

episode_count = 0
cur_rew  = np.zeros(NUM_ENVS)
cur_spd  = [[] for _ in range(NUM_ENVS)]
cur_col  = np.zeros(NUM_ENVS, dtype=bool)
cur_len  = np.zeros(NUM_ENVS, dtype=int)
ep_rew_window   = deque(maxlen=100)
ep_len_window   = deque(maxlen=100)
ep_crash_window = deque(maxlen=100)       # 1 = that episode ended in a crash

_live.update({k: [] for k in _live})
LAP_HISTORY.clear()
os.makedirs("models_racing_line", exist_ok=True)
BEST = {"lap_time": None, "step": None}


def eval_lap(step):
    """Time one deterministic lap, keep it for the plots, checkpoint the best, log to WandB."""
    lap = run_lap(eval_env, greedy_policy(net))
    LAP_HISTORY.append({"step": step, **lap})
    if lap["lap_time"] is not None and (BEST["lap_time"] is None or lap["lap_time"] < BEST["lap_time"]):
        BEST.update(lap_time=lap["lap_time"], step=step)
        torch.save(net.state_dict(), "models_racing_line/ppo_scratch_racing_line_best.pt")
    log = {"eval/lap_distance": lap["dist"], "eval/lap_completed": float(lap["lap_time"] is not None),
           "global_step": step}
    if lap["lap_time"] is not None:
        log["eval/lap_time"] = lap["lap_time"]
    wandb.log(log)
    return lap


eval_lap(0)                                  # lap of the untrained policy

obs = torch.tensor(train_env.reset(), dtype=torch.float32, device=device)
global_step    = 0
last_eval_step = 0
n_updates      = TOTAL_STEPS // (N_STEPS * NUM_ENVS)
print(f"Training {TOTAL_STEPS:,} decisions = {n_updates} updates; plots update below\n")

for update in range(n_updates):
    # ── Learning-rate annealing: linear from LR down to 0 ────────────────────
    lr_now  = LR * (1.0 - update / n_updates)
    ent_now = ENT_COEF * (1.0 - update / n_updates)      # explore early, commit to a line late
    for g in opt.param_groups:
        g["lr"] = lr_now

    mb_obs, mb_act, mb_logp, mb_val, mb_rew, mb_done = [], [], [], [], [], []
    update_speeds = []

    for _ in range(N_STEPS):
        action, logp, value = net.act(obs)
        clipped_np = np.clip(action.cpu().numpy(), -1.0, 1.0)
        next_obs, reward, dones, infos = train_env.step(clipped_np)
        raw_reward = train_env.get_original_reward()

        mb_obs.append(obs); mb_act.append(action); mb_logp.append(logp); mb_val.append(value)
        mb_rew.append(torch.tensor(reward, dtype=torch.float32, device=device))
        mb_done.append(torch.tensor(dones, dtype=torch.float32, device=device))

        cur_rew += raw_reward
        cur_len += 1
        for i, inf in enumerate(infos):
            cur_spd[i].append(inf.get("speed", 0.0))
            update_speeds.append(inf.get("speed", 0.0))
            if inf.get("collision", False) or inf.get("spun_out", False): cur_col[i] = True

        # Bootstrap truncated (not crashed) episodes before GAE
        for i, d in enumerate(dones):
            if d and infos[i].get("TimeLimit.truncated", False):
                term_obs = torch.tensor(infos[i]["terminal_observation"], dtype=torch.float32,
                                        device=device).unsqueeze(0)
                with torch.no_grad():
                    _, term_value = net(term_obs)
                mb_rew[-1][i] += GAMMA * term_value.item()

        for i, d in enumerate(dones):
            if d:
                episode_count += 1
                ep_rew_window.append(cur_rew[i]); ep_len_window.append(cur_len[i])
                ep_crash_window.append(float(cur_col[i]))
                wandb.log({
                    "racing/avg_speed":   float(np.mean(cur_spd[i])) if cur_spd[i] else 0.0,
                    "racing/crash_rate_last100": float(np.mean(ep_crash_window)),
                    "racing/episode_count": episode_count,
                    "rollout/ep_rew_mean":  float(np.mean(ep_rew_window)),
                    "rollout/ep_len_mean":  float(np.mean(ep_len_window)),
                    "global_step":          global_step,
                })
                cur_rew[i] = 0.0; cur_spd[i] = []; cur_col[i] = False; cur_len[i] = 0

        obs = torch.tensor(next_obs, dtype=torch.float32, device=device)
        global_step += NUM_ENVS

    with torch.no_grad():
        _, last_value = net(obs)
    values = torch.stack(mb_val)
    advantages, returns = compute_gae(torch.stack(mb_rew), values, torch.stack(mb_done),
                                      last_value, gamma=GAMMA, lam=GAE_LAMBDA)
    b_obs     = torch.stack(mb_obs).reshape(-1, obs_dim)
    b_act     = torch.stack(mb_act).reshape(-1, act_dim)
    b_logp    = torch.stack(mb_logp).reshape(-1)
    b_adv     = advantages.reshape(-1)
    b_ret     = returns.reshape(-1)
    b_val_old = values.reshape(-1)

    batch_size = b_obs.shape[0]
    idxs = np.arange(batch_size)
    epoch_losses, epoch_pg, epoch_vf = [], [], []
    for epoch in range(N_EPOCHS):
        np.random.shuffle(idxs)
        for start in range(0, batch_size, MINIBATCH):
            mb = idxs[start:start + MINIBATCH]
            dist, value = net(b_obs[mb])
            new_logp = dist.log_prob(b_act[mb]).sum(-1)
            entropy  = dist.entropy().sum(-1).mean()

            mb_adv = b_adv[mb]
            mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)

            ratio       = torch.exp(new_logp - b_logp[mb])
            surr1       = ratio * mb_adv
            surr2       = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * mb_adv
            policy_loss = -torch.min(surr1, surr2).mean()
            value_loss  = ((value - b_ret[mb]) ** 2).mean()
            loss        = policy_loss + VF_COEF * value_loss - ent_now * entropy

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), MAX_GRAD)
            opt.step()

            epoch_losses.append(loss.item())
            epoch_pg.append(policy_loss.item())
            epoch_vf.append(value_loss.item())

    y_true, y_pred = b_ret.cpu().numpy(), b_val_old.cpu().numpy()
    var_y = np.var(y_true)
    explained_var = float("nan") if var_y == 0 else 1.0 - np.var(y_true - y_pred) / var_y
    current_std   = net.log_std.exp().mean().item()
    wandb.log({
        "train/loss":                 float(np.mean(epoch_losses)),
        "train/policy_gradient_loss": float(np.mean(epoch_pg)),
        "train/value_loss":           float(np.mean(epoch_vf)),
        "train/std":                  current_std,
        "train/explained_variance":   explained_var,
        "train/learning_rate":        lr_now,
        "global_step":                global_step,
    })

    # ── Timed eval lap (builds the path-evolution plots) ──────────────────────
    if global_step - last_eval_step >= EVAL_FREQ or update == n_updates - 1:
        last_eval_step = global_step
        lap = eval_lap(global_step)
        print(f"  [eval lap at step {global_step:,}] {lap_str(lap)}")

    if episode_count > 0:
        _live["steps"].append(global_step)
        _live["rew"].append(float(np.mean(ep_rew_window)))
        _live["crash"].append(float(np.mean(ep_crash_window)))
        _live["speed"].append(float(np.mean(update_speeds)) if update_speeds else 0.0)
        _live["loss"].append(float(np.mean(epoch_losses)))
        _live["std"].append(current_std)
        draw_live_plots(TOTAL_STEPS)
        best_txt = f"{BEST['lap_time']:.2f} s" if BEST["lap_time"] else "none yet"
        print(f"Update {update+1}/{n_updates} | step {global_step:,} | episodes={episode_count} | "
              f"ep_rew_mean={np.mean(ep_rew_window):.1f} | crash rate={np.mean(ep_crash_window):.2f} | "
              f"lr={lr_now:.1e} | best lap={best_txt}")

torch.save(net.state_dict(), "models_racing_line/ppo_scratch_racing_line.pt")
train_env.save("models_racing_line/vecnormalize.pkl")
np.save("models_racing_line/lap_history.npy", np.array(LAP_HISTORY, dtype=object), allow_pickle=True)
print("\nSaved: models_racing_line/ppo_scratch_racing_line.pt       (final network)")
print(f"Saved: models_racing_line/ppo_scratch_racing_line_best.pt  (fastest eval lap: "
      f"{BEST['lap_time']:.2f} s at step {BEST['step']:,})" if BEST["lap_time"] else
      "No eval lap finished, so no best checkpoint was saved.")
print("Saved: models_racing_line/vecnormalize.pkl")
print("Saved: models_racing_line/lap_history.npy                  (every eval lap, for the plots)")
wandb.finish()
train_env.close()

# ── The best eval lap of this training run, on the map ───────────────────────
best_lap = min([h for h in LAP_HISTORY if h["lap_time"] is not None],
               key=lambda h: h["lap_time"], default=None)
if best_lap is not None:
    fig, ax = plt.subplots(figsize=(9, 6))
    track_ax(ax); lc = speed_line(ax, best_lap["path"], lw=2.2)
    ax.set_title(f"Best eval lap: {best_lap['lap_time']:.2f} s (step {best_lap['step']:,})")
    fig.colorbar(lc, ax=ax, fraction=0.03, label="speed (m/s)")
    plt.show()

---

# **Section 7 — Save Your Trained Model** 💾

The archive holds the best and final policy weights, the reward-normalization stats and every
eval lap (`lap_history.npy`), so you can regenerate all the plots later without retraining.

In [ ]:
!zip -r models_racing_line_archive.zip /content/models_racing_line

In [ ]:
from google.colab import files
files.download('models_racing_line_archive.zip')

---

# **Section 8 — Evaluation & Visualization** 🎬

## 8.1 Final Lap vs the Baselines 🏁

We load the **best-lap checkpoint**, time it from the start line, and compare it with the
centerline baselines from Section 4.3. Then we check that it is not a one-lap fluke: **10 runs
of 30 s from random points on the track**, counting crashes.

In [ ]:
# ── Final lap vs baselines + robustness check ─────────────────────────────────
ckpt = ("models_racing_line/ppo_scratch_racing_line_best.pt"
        if os.path.exists("models_racing_line/ppo_scratch_racing_line_best.pt")
        else "models_racing_line/ppo_scratch_racing_line.pt")
eval_net = ActorCritic(obs_dim, act_dim).to("cpu")
eval_net.load_state_dict(torch.load(ckpt, map_location="cpu"))
eval_net.eval()
print(f"Loaded {ckpt}\n")

FINAL = run_lap(eval_env, greedy_policy(eval_net))
RESULTS = {**{k: v for k, v in BASELINES.items() if k != "Reference raceline"},   # centerline baselines
           "RL racing line (ours)": FINAL}

print("─" * 66)
print(f"  {'Lap':<28} {'Lap time':>16} {'Avg speed':>10} {'Top speed':>10}")
print("─" * 66)
for name, r in RESULTS.items():
    print(f"  {name:<28} {lap_str(r):>16} {np.mean(r['path'][:, 2]):>6.2f} m/s {np.max(r['path'][:, 2]):>6.2f} m/s")
print("─" * 66)

# Robustness: 10 x 30 s runs from random start points (the training reset)
crashes, speeds = 0, []
policy = greedy_policy(eval_net)
for k in range(10):
    obs, _ = eval_env.reset(seed=1000 + k)
    done = trunc = False
    while not (done or trunc):
        obs, _, done, trunc, info = eval_env.step(policy(obs))
        speeds.append(info["speed"])
    crashes += int(info["collision"] or info["spun_out"])
print(f"\nRobustness: {10 - crashes}/10 random-start runs of 30 s without a crash "
      f"(average speed {np.mean(speeds):.2f} m/s)")

names = [n for n, r in RESULTS.items() if r["lap_time"]]
times = [RESULTS[n]["lap_time"] for n in names]
fig, ax = plt.subplots(figsize=(10, 3.5))
bars = ax.barh(names, times, color=["#9e9e9e"] * (len(names) - 1) + ["#ff9800"])
ax.bar_label(bars, fmt="%.2f s", padding=4)
ax.set_xlabel("lap time (s), lower is better"); ax.invert_yaxis()
ax.set_xlim(0, max(times) * 1.15); ax.set_title("Lap time comparison")
plt.tight_layout(); plt.show()

## 8.2 How the Racing Line Evolved 🧬

Four eval laps from across training (the first ones may be DNF laps that spun out), drawn on the map and coloured by speed. The bottom row
zooms into the sharpest corner (C1), with the **centerline in gray** and the **reference
raceline in cyan** (the line that ships with the map files) for comparison. Watch for:
**later braking**, **a smoother arc through the corner**, and **higher speed on the straights**.

In [ ]:
# ── Path evolution: 4 snapshots across training ───────────────────────────────
pool  = [h for h in LAP_HISTORY if len(h["path"]) > 1]          # every eval lap, DNF laps included
picks = [pool[0], pool[len(pool) // 3], pool[2 * len(pool) // 3], pool[-1]] if len(pool) >= 4 else pool

fig, axs = plt.subplots(2, len(picks), figsize=(5 * len(picks), 8.5),
                        gridspec_kw={"height_ratios": [1, 1.1]}, squeeze=False)
for k, h in enumerate(picks):
    track_ax(axs[0, k]); lc = speed_line(axs[0, k], h["path"], lw=2.0)
    axs[0, k].set_title(f"step {h['step']:,}\nlap {lap_str(h)}")
    track_ax(axs[1, k], CORNER_XY[0], zoom=8)
    plain_line(axs[1, k], CL_XY, CL_COLOR, "centerline", ls="--", lw=1.2)
    plain_line(axs[1, k], RL_XY, RL_COLOR, "reference raceline", lw=1.5)
    speed_line(axs[1, k], h["path"], lw=3)
    axs[1, k].set_title("corner C1")
axs[1, 0].legend(loc="lower left", fontsize=9)
fig.colorbar(lc, ax=axs, fraction=0.015, label="RL speed (m/s)")
plt.subplots_adjust(hspace=0.15)
plt.show()

## 8.3 Racing-Line Evolution, Animated 🎞️

Every eval lap from training, one frame each: the full lap on the left and corner C1 on the
right. Saved as `racing_line_evolution.gif`.

In [ ]:
# ── Animated path evolution (GIF) ─────────────────────────────────────────────
from matplotlib.animation import FuncAnimation, PillowWriter

frames = [h for h in LAP_HISTORY if len(h["path"]) > 1]
fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(12, 5.5), gridspec_kw={"width_ratios": [1.6, 1]})


def draw_frame(i):
    h = frames[i]
    for ax in (ax_full, ax_zoom):
        ax.clear()
    track_ax(ax_full); speed_line(ax_full, h["path"], lw=2.0)
    track_ax(ax_zoom, CORNER_XY[0], zoom=8)
    plain_line(ax_zoom, CL_XY, CL_COLOR, ls="--", lw=1.2)
    plain_line(ax_zoom, RL_XY, RL_COLOR, lw=1.5)
    speed_line(ax_zoom, h["path"], lw=3)
    ax_full.set_title(f"training step {h['step']:,}: lap {lap_str(h)}")
    ax_zoom.set_title("corner C1 (gray = centerline, cyan = reference raceline)", fontsize=10)


anim = FuncAnimation(fig, draw_frame, frames=len(frames), interval=600)
anim.save("racing_line_evolution.gif", writer=PillowWriter(fps=2))
plt.close(fig)
print(f"Saved racing_line_evolution.gif ({len(frames)} frames)")
display(Image(filename="racing_line_evolution.gif"))

## 8.4 Lap Time Over Training 📉

In [ ]:
# ── Lap time vs training steps ────────────────────────────────────────────────
steps = [h["step"] for h in LAP_HISTORY if h["lap_time"] is not None]
laps  = [h["lap_time"] for h in LAP_HISTORY if h["lap_time"] is not None]
dnf   = [h["step"] for h in LAP_HISTORY if h["lap_time"] is None]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(steps, laps, "o-", color="#ff9800", lw=2, label="RL eval lap")
for (name, r), c in zip(BASELINES.items(), ["#9e9e9e", "#607d8b", RL_COLOR]):
    if r["lap_time"] and name != "Reference raceline":
        ax.axhline(r["lap_time"], color=c, ls="--", lw=1.4, label=f"{name} ({r['lap_time']:.1f} s)")
if dnf:
    ax.plot(dnf, [max(laps) if laps else 0] * len(dnf), "x", color="red", ms=9, mew=2, label="DNF eval lap")
if BEST["lap_time"]:
    ax.plot([BEST["step"]], [BEST["lap_time"]], "*", color="gold", ms=16, mec="k", label="best-lap checkpoint")
ax.set_xlabel("training decisions"); ax.set_ylabel("lap time (s)")
ax.set_title("Lap time over training, lower is better"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8.5 Speed Profile and Line Position 🔍

Two views along the lap distance, for the untrained policy (centerline), the reference
raceline and our learned line:
- **Speed:** where each lap brakes and accelerates (the red bands are corners C1 to C3)
- **Lateral position:** where on the track width the car drives (+ is left of the centerline;
  the dashed lines are the track edges)

In [ ]:
# ── Speed + lateral position along the lap ────────────────────────────────────
_cl_t = np.column_stack([np.cos(CL.yaws), np.sin(CL.yaws)])      # centerline tangents


def along_track(path):
    """For each path point: distance along the centerline and signed lateral offset."""
    xy = path[:, :2]
    d2 = ((xy[:, None, :] - CL_XY[None, ::2, :]) ** 2).sum(-1)    # every 2nd centerline point
    j  = np.argmin(d2, axis=1) * 2
    rel = xy - CL_XY[j]
    lat = _cl_t[j, 0] * rel[:, 1] - _cl_t[j, 1] * rel[:, 0]        # + is left of the centerline
    return np.asarray(CL.ss)[j], lat


show   = {CENTER_KEY: BASELINES[CENTER_KEY], "Reference raceline": BASELINES["Reference raceline"],
          "RL racing line (ours)": FINAL}
colors = {CENTER_KEY: "#9e9e9e", "Reference raceline": RL_COLOR, "RL racing line (ours)": "#ff9800"}

fig, (ax_v, ax_y) = plt.subplots(2, 1, figsize=(14, 7.5), sharex=True)
for s0 in CORNER_S:
    for ax in (ax_v, ax_y):
        ax.axvspan(s0 - 6, s0 + 6, color="red", alpha=0.08)
for name, r in show.items():
    if len(r["path"]) < 2:
        continue
    s, lat = along_track(r["path"])
    order = np.argsort(s)
    ax_v.plot(s[order], r["path"][order, 2], color=colors[name], lw=1.6, label=name)
    ax_y.plot(s[order], lat[order], color=colors[name], lw=1.6, label=name)
for e in (HALF_WIDTH, -HALF_WIDTH):
    ax_y.axhline(e, color="k", ls="--", lw=1)
ax_v.set_ylabel("speed (m/s)"); ax_y.set_ylabel("lateral position (m)")
ax_y.set_xlabel("distance along the centerline (m)")
ax_v.set_title("Speed profile (red bands are corners C1 to C3)"); ax_v.legend(); ax_v.grid(alpha=0.3)
ax_y.set_title("Where on the track width each lap drives"); ax_y.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8.6 Corner by Corner: Our Line vs the Reference Raceline 🔬

The gray dashed line is the centerline, the cyan line is the reference raceline from the map
files, and the coloured line is our learned line (colour shows its speed).

In [ ]:
# ── Zoom on all 3 corners: centerline vs reference raceline vs RL line ────────
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
for k, ax in enumerate(axs):
    track_ax(ax, CORNER_XY[k], zoom=8)
    plain_line(ax, CL_XY, CL_COLOR, "centerline", ls="--", lw=1.2)
    plain_line(ax, BASELINES["Reference raceline"]["path"][:, :2], RL_COLOR, "reference raceline", lw=2)
    lc = speed_line(ax, FINAL["path"], lw=3.5)
    ax.set_title(f"Corner C{k+1}: RL line coloured by its speed")
axs[0].legend(loc="lower left")
fig.colorbar(lc, ax=axs, fraction=0.015, label="RL speed (m/s)")
plt.show()

## 8.7 Is the Learned Line Physically Sensible? 🧪

A lap time alone does not tell us *how* the car drives. The best check is the **sideways
(lateral) acceleration** the car pulls along the lap, measured exactly from the simulator
state as speed times yaw rate:

$$a_{\text{lat}} \;=\; v\,\omega$$

- The tyres can give at most about $\mu g$ (with the gym's friction $\mu = 1.05$, about 10.3 m/s²)
- Past 1.1 times that limit (about 11.3 m/s²) the episode ends as a spin-out, and the reward starts charging at 80% of it (about 8.2 m/s²)
- The trace is low-pass filtered over about 0.2 s, the same signal the reward and the spin-out rule use

A good, realistic racing line **uses most of the grip in the corners** (peaks close to the
dotted 80% line) and **never goes past the real tyre limit**, with smooth rises and falls.
We compare our line with the reference raceline.

**Why does the line run near the inside of some corners?** Where a corner can be taken at
the 10 m/s top speed without running out of grip, the **shortest** path is the fastest one,
so the line takes the inside. In the tight corners grip runs out first, so the line opens up
into a wider arc to carry more speed. The wall-margin penalty keeps a safety gap to the wall.

In [ ]:
# ── Lateral acceleration along the lap vs the tyre grip limit ─────────────────
fig, ax = plt.subplots(figsize=(14, 4.5))
for name, r, c in [("Reference raceline", BASELINES["Reference raceline"], RL_COLOR),
                   ("RL racing line (ours)", FINAL, "#ff9800")]:
    if len(r["path"]) < 50:
        continue
    p = r["path"][::5]                       # one point per decision is plenty
    s, _ = along_track(p)
    order = np.argsort(s)
    ax.plot(s[order], p[order, 3], color=c, lw=1.5,
            label=f"{name} (peak {np.max(p[:, 3]):.1f} m/s²)")
ax.axhline(MU_G, color="red", ls="--", lw=1.5, label=f"real tyre grip limit, about {MU_G:.1f} m/s²")
ax.axhline(SPIN_OUT_LIMIT, color="darkred", ls="-", lw=1, label=f"spin-out, {SPIN_OUT_LIMIT:.1f} m/s²")
ax.axhline(GRIP_THRESHOLD, color="darkorange", ls=":", lw=1.5, label=f"grip penalty starts (80%), {GRIP_THRESHOLD:.1f} m/s²")
for s0 in CORNER_S:
    ax.axvspan(s0 - 6, s0 + 6, color="red", alpha=0.08)
ax.set_xlabel("distance along the centerline (m)"); ax.set_ylabel("lateral acceleration (m/s²)")
ax.set_title("How much grip each line uses (red bands are corners C1 to C3)")
ax.set_ylim(0, SPIN_OUT_LIMIT * 1.2); ax.legend(loc="upper right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8.8 Record Eval Video 🎥

In [ ]:
# ── Record a video of the final racing line in the simulator ──────────────────
import imageio

vid_env = RacingLineWrapper(render_mode="rgb_array")
obs, _ = vid_env.reset(options={"poses": START_POSE})
policy = greedy_policy(eval_net)
frames, dist, n = [], 0.0, 0
while dist < vid_env.track_len and n < 3000:
    obs, _, done, _, info = vid_env.step(policy(obs))
    dist += info["ds"]; n += 1
    frame = vid_env.render()                    # one frame per decision (20 fps of driving)
    if frame is not None:
        frames.append(frame)
    if info["collision"] or info["spun_out"]:
        print("Crashed during the video lap"); break
vid_env.close()

if frames:
    imageio.mimsave("eval_render_racing_line.mp4", frames, fps=20)
    print(f"✅ Captured {len(frames)} frames ({n * ACTION_REPEAT * vid_env.dt:.1f} s of driving)")
    from google.colab import files
    files.download("eval_render_racing_line.mp4")
else:
    print("❌ No frames captured")

---

# **Section 9 — Summary** 📝

## What We Built

1. **A new job for the policy:** instead of steering, it chooses *where to be on the track*
   and *how fast to go*, and pure pursuit turns that into steering. The path it drives **is**
   the learned racing line.
2. **A lap-time objective:** fixed-length episodes rewarded by metres of centerline progress,
   so more progress means a faster lap, with only a small wall-margin term and no centerline
   or speed shaping to bias the line.
3. **Planning ahead:** 20 Hz decisions (about a 5 s horizon with $\gamma = 0.99$) and a 20 m
   curvature preview let the policy brake *before* corners instead of reacting in them.
4. **Real-car physics through the reward:** exceeding the tyres' grip is a spin-out
   (a crash), and riding above 80% of grip costs reward, so the policy itself learns lines a
   real car could drive, which pushes it toward the classic wide-entry, apex, wide-exit shape.
5. **Stable training:** learning-rate and entropy annealing and a best-lap checkpoint, so a late bad
   update cannot throw away the best line.
6. **Proof that it is a real line:** every eval lap was recorded, so we watched the line
   evolve, compared it corner by corner with the reference raceline, checked it on random
   starts, and confirmed it stays within the tyres' grip limit.

The PPO algorithm (network, GAE, clipped loss) is the same one we built in Learn to Race.
Once again, the new capability came entirely from **how we framed the problem**: the action
space, the reward and the observation.

## Things to Try

| Experiment | What to look for |
|---|---|
| Set `ACTION_REPEAT = 1` | Does the policy still learn to brake before the hairpin? |
| Remove the curvature preview (set it to 0.5) | How much slower does it get in the corners? |
| Raise `V_MAX` to 15 m/s | More of the lap becomes grip-limited: does the line widen in more corners? |
| Set `WALL_PROX_COEF = 0` | Does the line move closer to the walls, and do crashes go up? |
| Set `GRIP_THRESHOLD = MU_G` (no grip reserve) | How much lap time does the 20% margin cost, and do spin-outs go up? |
| Train on a different map | Does the same reward find a sensible line elsewhere? |